# Part 6: 모든 소스를 한데 모으기 (Work IQ 포함)

COO가 전사 회의 전에 빠른 회사 현황 점검을 요청했습니다. Fabric의 재고 위험, 재고 상황에 대한 활성 이메일 커뮤니케이션 요약, 회사 정책의 에스컬레이션 프로토콜, 직원 대상 건강 복리후생 하이라이트, 그리고 업계 벤치마크까지 필요합니다. 모두 하나의 답변으로 말이죠.

**🎯 미션**
- HR 문서, 건강 문서, Fabric IQ, Work IQ, Web IQ를 결합한 **5개 소스 지식 베이스** 구축하기
- 구조화된 제품 데이터, 개인 업무 커뮤니케이션, 정책 문서, 건강 복리후생, 실시간 웹 벤치마크를 동시에 아우르는 단일 질의 실행하기

## 워크플로우

<img src="../img/workflow-part6.png" width="700"/>


## 1단계: 환경 변수 설정

Azure AI Search, Azure OpenAI, Fabric, Web IQ 리소스에 대한 구성을 로드합니다. 프로젝트 폴더의 `.env` 파일에 이 파트에 필요한 값이 들어 있습니다.

‼️ `WEB_IQ_KEY`가 아직 설정되어 있지 않다면, Part 2에서 안내한 대로 `.env` 파일의 `WEB_IQ_KEY=your-web-iq-key`를 실제 키 값으로 교체해 주세요.

In [ ]:
import json
import os

from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
FABRIC_WORKSPACE_ID = os.environ["FABRIC_WORKSPACE_ID"]
FABRIC_ONTOLOGY_ID = os.environ["FABRIC_ONTOLOGY_ID"]
WEB_IQ_KEY = os.environ["WEB_IQ_KEY"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = DefaultAzureCredential()

print("Environment variables loaded")

## 2단계: 로그인 및 위임 토큰 획득

지식 베이스 검색이 Work IQ 및 Fabric IQ 지식 소스와 함께 여러분의 ID를 사용하려면, Microsoft 365 데이터와 Fabric 워크스페이스 양쪽에 액세스 권한이 있는 로그인된 사용자의 OAuth 토큰이 필요합니다.

아래 셀을 실행하여 안내 사이드바의 자격 증명으로 Azure에 로그인하세요. 터미널에 표시되는 URL과 코드를 브라우저에서 입력해 로그인합니다(Codespaces 등 원격 환경에서도 동작). 성공하면 Azure AI Search 범위에 대한 위임 토큰을 획득하고 검색 중 `query_source_authorization`으로 사용하도록 저장합니다.

In [ ]:
from azure.identity import DeviceCodeCredential

user_credential = DeviceCodeCredential(tenant_id=AZURE_TENANT_ID)
user_token = user_credential.get_token("https://search.azure.com/.default").token

print("Acquired token for logged-in user")

### 데모 이메일 확인

Part 4에서 이미 본인 메일함에 데모 이메일 3통을 시딩했다면, 같은 메일함을 사용하는 이 파트에서는 다시 보낼 필요가 없습니다. 아직 Part 4를 진행하지 않았다면, 먼저 [Part 4 노트북](part4-work-iq-to-kb.ipynb)의 "본인 메일함에 데모 이메일 시딩하기" 셀을 실행한 뒤 이어서 진행해 주세요.


## 3단계: 모든 지식 소스 만들기

이 지식 베이스를 위해 다섯 개의 지식 소스를 만듭니다. 앞에서 사용한 것과 동일한 두 개의 검색 인덱스 소스에 더해, Work IQ, Fabric 온톨로지, Web IQ 소스입니다.

* `hrdocs-knowledge-source`: `hrdocs` 검색 인덱스를 가리킴
* `healthdocs-knowledge-source`: `healthdocs` 검색 인덱스를 가리킴
* `workiq-knowledge-source`: 업무 맥락을 위해 Work IQ를 가리킴
* `fabric-ontology-knowledge-source`: 구조화된 운영 데이터에 사용되는 Fabric 워크스페이스와 온톨로지를 가리킴
* `web-knowledge-source`: 키 기반 인증과 `web` 도구를 사용해 Web IQ 서비스의 원격 MCP 서버를 가리킴

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    FabricOntologyKnowledgeSource,
    FabricOntologyKnowledgeSourceParameters,
    McpServerAutoOutputParsing,
    McpServerKnowledgeSource,
    McpServerKnowledgeSourceParameters,
    McpServerStoredHeadersAuthentication,
    McpServerStoredHeadersParameters,
    McpServerTool,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    WorkIQKnowledgeSource,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
WORK_KNOWLEDGE_SOURCE_NAME = "workiq-knowledge-source"
FABRIC_KNOWLEDGE_SOURCE_NAME = "fabric-ontology-knowledge-source"
WEB_KNOWLEDGE_SOURCE_NAME = "web-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    HR_KNOWLEDGE_SOURCE_NAME,
    HEALTH_KNOWLEDGE_SOURCE_NAME,
    WORK_KNOWLEDGE_SOURCE_NAME,
    FABRIC_KNOWLEDGE_SOURCE_NAME,
    WEB_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Zava HR 문서"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Zava 건강 복리후생 문서"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

work_knowledge_source = WorkIQKnowledgeSource(
    name=WORK_KNOWLEDGE_SOURCE_NAME,
    description="Zava Work IQ 지식 소스",
)
index_client.create_or_update_knowledge_source(knowledge_source=work_knowledge_source)

fabric_knowledge_source = FabricOntologyKnowledgeSource(
    name=FABRIC_KNOWLEDGE_SOURCE_NAME,
    description="Zava Fabric Ontology 지식 소스",
    fabric_ontology_parameters=FabricOntologyKnowledgeSourceParameters(
        workspace_id=FABRIC_WORKSPACE_ID,
        ontology_id=FABRIC_ONTOLOGY_ID,
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=fabric_knowledge_source)

web_knowledge_source = McpServerKnowledgeSource(
    name=WEB_KNOWLEDGE_SOURCE_NAME,
    description="Web IQ(실시간 웹 검색)",
    mcp_server_parameters=McpServerKnowledgeSourceParameters(
        server_url="https://api.microsoft.ai/v3/mcp",
        authentication=McpServerStoredHeadersAuthentication(
            stored_headers_parameters=McpServerStoredHeadersParameters(
                {"headers": {"x-apikey": WEB_IQ_KEY}}
            )
        ),
        tools=[McpServerTool(name="web", output_parsing=McpServerAutoOutputParsing())],
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=web_knowledge_source)
print("Knowledge sources created")

## 4단계: 결합된 지식 베이스 만들기

지식 베이스는 지식 소스를 LLM 및 검색 동작 방식에 대한 지침과 결합합니다.

이 노트북에서 지식 베이스는 연결된 모델이 질문에 직접 답할 수 있도록 `answerSynthesis` `outputMode`를 사용합니다. 또한 모델이 질의 계획과 소스 선택을 도울 수 있도록 `low` `retrievalReasoningEffort`를 사용합니다.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-work-fabric-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="HR 및 건강 문서를 위한 검색 인덱스, 업무 맥락을 위한 Work IQ, 제품 데이터를 위한 Fabric Ontology, 실시간 웹 검색을 위한 Web IQ를 결합한 Zava KB",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions=(
        "업무 맥락에는 Work IQ를, 구조화된 운영 데이터에는 Fabric Ontology를, HR 및 건강 정책 문서에는 검색 인덱스를, 공개 웹 벤치마크가 필요하면 Web IQ를 사용하세요. "
        "Fabric Ontology에는 복합 질문 전체를 보내지 말고 재고 관련 하위 질문만 분리해 보내세요. "
        "Fabric Ontology 질의는 반드시 엔터티와 필드, 집계 방식을 명시한 영어 문장으로 작성하세요. "
        "예: 'Using Product entities, group by category, calculate the sum of stockLevel for each category, "
        "and return the category with the lowest total stockLevel.'"
    ),
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")

## 5단계: 지식 베이스에 질의

다섯 개의 소스를 아우르는 다섯 부분으로 된 전사 회의 질문을 합니다.

* _"현재 재고 수준이 가장 낮은 제품 카테고리는 무엇인가요?"_ → **Fabric IQ**
* _"클로 해머 재고 상황에 대한 최근 이메일이 있나요?"_ → **Work IQ**
* _"예산 관리를 담당하는 회사 직무는 무엇인가요?"_ → `hrdocs`
* _"Zava는 직원에게 어떤 정신 건강 보장을 제공하나요?"_ → `healthdocs`
* _"소매업체들의 일반적인 재고 회전율 벤치마크는 무엇인가요?"_ → **Web IQ**

ℹ️ Fabric IQ는 다른 주제와 섞인 복합 질문에서 간헐적으로 질의를 거부하는 경우가 있어, Fabric IQ 질문만 별도 호출로 분리해 안정적으로 답을 받고 나머지 네 질문은 한 번의 호출로 처리합니다.

⏱️ Work IQ, Fabric IQ, Web IQ 지식 소스의 증가된 지연 시간 때문에 샘플 질문에 답하는 데 약 1~2분이 걸립니다.


In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FabricOntologyKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    KnowledgeSourceParams,
    SearchIndexKnowledgeSourceParams,
    WorkIQKnowledgeSourceParams,
)
from IPython.display import Markdown, display
import time

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

# Fabric IQ (Preview) 의 NL2Ontology 파서는 다른 주제와 섞인 복합 질문에서 간헐적으로 실패하는 경우도 있어
# Fabric IQ에는 단독 질문형태로 보내고 나머지 소스는 별도 호출로 분리해 신뢰성을 확보합니다.
fabric_question = "현재 재고 수준이 가장 낮은 제품 카테고리는 무엇인가요?"
other_sources_question = ("금요일 전체 회의입니다: "
                           "2) 프로페셔널 클로 해머 재고에 대한 최근 업무 이메일이 있나요? "
                           "3) 예산 관리를 담당하는 Zava 직무는 무엇인가요? "
                           "4) Zava는 직원에게 어떤 정신 건강 보장을 제공하나요? 전체 회의에서 복리후생을 간단히 언급하고 싶습니다. "
                           "5) 소매업체들의 일반적인 재고 회전율(inventory turnover) 벤치마크는 무엇인가요? 웹에서 최신 자료를 찾아 주세요. "
                           "답변도 2) 3) 4) 5) 번호를 그대로 사용해 주세요.")

fabric_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text=fabric_question)]),
    ],
    knowledge_source_params=[
        FabricOntologyKnowledgeSourceParams(
            knowledge_source_name=FABRIC_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True,
            reranker_threshold=0,
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=120,
)

other_sources_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text=other_sources_question)]),
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        WorkIQKnowledgeSourceParams(
            knowledge_source_name=WORK_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
        ),
        KnowledgeSourceParams(
            knowledge_source_name=WEB_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            kind="mcpServer",
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=300,
)

# 단독 질문도 드물게 실패할 수 있으므로 Fabric IQ 호출만 짧게 재시도합니다.
MAX_FABRIC_ATTEMPTS = 2
fabric_result = None
fabric_failed = True
for attempt in range(1, MAX_FABRIC_ATTEMPTS + 1):
    fabric_result = knowledge_base_client.retrieve(
        retrieval_request=fabric_request,
        query_source_authorization=user_token,
    )
    fabric_failed = any(
        getattr(activity, "type", None) == "fabricOntology" and getattr(activity, "error", None)
        for activity in (fabric_result.activity or [])
    )
    if not fabric_failed:
        break
    print(f"⚠️ Fabric IQ 질의가 실패했습니다 (시도 {attempt}/{MAX_FABRIC_ATTEMPTS}). 재시도 중...")
    time.sleep(3)

other_result = knowledge_base_client.retrieve(
    retrieval_request=other_sources_request,
    query_source_authorization=user_token,
)

# 활동 로그/참조 검토 셀에서 두 호출 결과를 모두 볼 수 있도록 하나로 병합
result = other_result
result.activity = (fabric_result.activity or []) + (other_result.activity or [])
result.references = (fabric_result.references or []) + (other_result.references or [])

fabric_answer = (
    fabric_result.response[0].content[0].text if not fabric_failed else "재고 카테고리 데이터를 가져오지 못했습니다."
)
answer_text = f"1) {fabric_answer}\n\n" + other_result.response[0].content[0].text

# 응답 메시지에서 합성된 답변 추출
display(Markdown(answer_text))

### 활동 로그 검토

이 활동 로그에서는 "fabricOntology", "workIQ", "searchIndex", "mcpServer"에 대한 호출과 각 단계의 질의를 볼 수 있습니다.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2, ensure_ascii=False))

### 참조 검토

지식 베이스의 여러 소스에서 온 다양한 참조 유형이 혼합되어 보일 것입니다.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2, ensure_ascii=False))

#### 🔍 소스 헌트

위에 출력된 참조를 살펴보세요. 다섯 가지 소스 유형을 모두 식별할 수 있나요?

1. 어떤 참조가 **Fabric IQ**(구조화된 제품 엔터티)에서 왔나요?
2. 어떤 참조가 **Work IQ**(여러분의 M365 이메일/Teams)에서 왔나요?
3. 어떤 참조가 **HR 인덱스**(정책 문서)에서 왔나요?
4. 어떤 참조가 **healthdocs**(건강 복리후생 문서)에서 왔나요?
5. 어떤 참조가 **Web IQ**(실시간 웹 검색)에서 왔나요?

## ✅ 미션 완료

**워크샵 전체에서 만든 것:**

* ✓ **5개 소스 지식 베이스**: HR 문서, 건강 문서, 구조화된 Fabric 제품 데이터, 라이브 M365 업무 커뮤니케이션, 실시간 웹 검색에 걸친 단일 지식 베이스.
* ✓ **병렬 멀티 소스 검색**: 하나의 질문이 하위 질의로 분해되어 다섯 가지 소스 유형 모두로 동시에 라우팅됨.
* ✓ **완전한 에이전트형 RAG 파이프라인**: 미리 구축된 인덱스부터 Fabric 온톨로지, 개인 이메일, 실시간 웹까지, 지식 베이스가 하나의 응답에서 모든 소스를 인용하고 합성함.